# 02 · System A — cross-attention fusion of text and physiology

A **trained** system with one independent head per subtask, optimized directly
on the soft (disagreement-aware) targets. Two streams meet in a bidirectional
cross-attention block:

```
 capsanocr text ─▶ Transformer encoder ─▶ mean-pool ─▶ z_text
                                                        │
 EEG · HR · eye-tracking ─▶ PhysioEncoder ─▶ z_physio    │
     flat   : MLP over the aggregated vector  (memes)    │
     matrix : shared subject MLP + attention pool (videos)
                                                        ▼
                              ┌───────────────────────────────────┐
                              │  text ⇄ physio cross-attention    │
                              │  residual + LayerNorm + concat    │
                              └─────────────────┬─────────────────┘
                                                ▼
                     x.1: 2 logits · x.2: 3 logits · x.3: 6 logits
```

**Why cross-attention rather than concatenation.** Each modality should
*reinterpret* the other: a strong physiological response ought to change how an
ambiguous text is read, and a blunt text ought to change how a weak response is
read. A flat concatenation cannot express that modulation.

**What this notebook is for.** Reading the design and inspecting one run. Long
searches and final training belong on the command line:

```bash
python scripts/run_fusion.py --config fusion_memes  --steps hpo,train
python scripts/run_fusion.py --config fusion_videos --steps train,submit
```

In [ ]:
# Makes the notebook work from a clone (no install) and on Colab alike.
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

# Point this at the directory holding the EXIST 2026 corpus. The configs read
# it from here, so nothing below contains a hard-coded path.
import os
os.environ.setdefault("EXIST2026_ROOT", str(Path.home() / "EXIST_2026"))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1 · Configuration

`MODALITY` selects everything that differs between the two: the physiological
representation, the text encoder, and the token window.

|            | memes                        | videos                                  |
|------------|------------------------------|-----------------------------------------|
| physiology | `flat` — aggregated vector   | `matrix` — per subject + attention pool |
| encoder    | XLM-R base (bilingual)       | emotion-tuned RoBERTa large             |
| max tokens | 256                          | 512                                     |
| split      | stratified by (lang, x.1)    | grouped by TikTok creator               |

In [ ]:
MODALITY = "videos"   # "memes" | "videos"

from exist2026.config import load_fusion_config
from exist2026.fusion.training import device_of

config = load_fusion_config(f"fusion_{MODALITY}")
config.dirs.create()
DEVICE = device_of()

print(f"modality : {config.modality.value}")
print(f"physio   : {config.physio_kind}")
print(f"encoder  : {config.text_model} (max_len={config.max_len})")
print(f"device   : {DEVICE}")
print(f"outputs  : {config.dirs.root}")

## 2 · Build the experiment

`FusionExperiment.build` does the whole assembly in the one order that is safe:
load records → soft labels → discover the physiological feature schema →
**split** → *then* fit the preprocessor on the training side only. Fitting the
KNN imputer or the z-score before the split would leak validation statistics
into training, and the leak would be invisible in the metrics.

In [ ]:
from exist2026.fusion.experiment import FusionExperiment

experiment = FusionExperiment.build(config)
experiment.split.summary()

### The enriched text view

`capsanocr` = a neutral visual description (`cap`) + an independent
gender-relevance analysis (`san`) + the on-screen text (`ocr`). The description
and the analysis come from **two separate prompts**: keeping the sexism
reasoning in its own generation is what stops the description from being
written to fit a conclusion the model already reached.

In [ ]:
example_id = experiment.split.train[0]
print(experiment.text_view(example_id)[:900], "\n[...]")

### The soft targets

The class order below is fixed across the losses, the model heads and the
PyEvALL writers. Note that x.3 does **not** sum to one: the categories are
independent probabilities, which is why it is trained with BCE.

In [ ]:
import numpy as np

for subtask_key in config.subtask_keys:
    print(f"{subtask_key}: {np.round(experiment.soft[example_id][subtask_key], 3).tolist()}")

### The physiological branch

Each stimulus was seen by only 2-4 subjects, so missing subjects and missing
features are the normal case, not an error path. `matrix` keeps one row per
subject plus a validity mask and lets the model pool them with attention, which
is permutation-invariant and masks absent subjects out instead of averaging
them in.

In [ ]:
values = experiment.raw_physio[example_id]
print(f"feature schema : {experiment.schema.dim} features "
      f"(ET={len(experiment.schema.eye_tracking)} "
      f"HR={len(experiment.schema.heart_rate)} "
      f"EEG={len(experiment.schema.eeg)})")
print(f"representation : {config.physio_kind}, shape {values.shape}")
if config.physio_kind == "matrix":
    print(f"valid subjects : {int(experiment.physio_masks[example_id].sum())} of {config.max_subjects}")
print(f"missing values : {int(np.isnan(values).sum())} before imputation")

## 3 · The model

One model per subtask, built from a hyperparameter dict. The text backbone is
frozen by default — the head has orders of magnitude fewer examples than the
encoder has parameters.

In [ ]:
from exist2026.fusion.modeling import build_model, trainable_parameters

key = config.subtask_keys[0]
model = build_model(key, config, config.training.merged(), experiment.physio_in).to(DEVICE)

print(model.__class__.__name__, f"| head width = {model.n_out}")
print(f"trainable parameters: {trainable_parameters(model) / 1e6:.2f}M")
print(f"fusion output dim   : {model.fusion.out_dim}")
del model

## 4 · Training one subtask

Two things worth stating explicitly, because they are easy to get wrong:

- **Checkpoints are selected by ICM-Soft-Norm**, the metric the lab ranks on,
  computed on the *full* validation split with the hierarchical gate applied.
  Not by validation loss, and not by F1 — a model can improve either while
  getting worse at the metric that decides the ranking. The F1 in the log is a
  sanity signal only.
- **x.2 and x.3 train on sexist instances only but are evaluated on
  everything.** Training them on non-sexist instances would teach a NO class
  the deployed cascade never asks about; evaluating them only on sexist ones
  would hide the cost of the gate's mistakes.

In [ ]:
from exist2026.fusion.training import train_task

batch_size = config.training.merged()["batch_size"]

identification = train_task(
    key,
    config=config,
    physio_in=experiment.physio_in,
    train_loader=experiment.train_loader(key, batch_size, DEVICE),
    val_loader=experiment.val_loader(device=DEVICE),
    gold=experiment.gold(key),
    checkpoint_name=f"best_{key}.pt",
    device=DEVICE,
)
print(f"best ICM-Soft-Norm: {identification.best_metric:.4f}")

## 5 · Hyperparameter search (optional, slow)

Sequential over the hierarchy: tune x.1, retrain its best configuration and
freeze it as the gate, then tune x.2 and x.3 *through that gate*. Tuning them
against a moving gate would optimize each subtask against a model that will not
exist at inference time.

TPE proposes, the median pruner kills unpromising trials after the first epoch.
Prefer `scripts/run_fusion.py --steps hpo` for a full search.

In [ ]:
RUN_SEARCH = False   # set True to search from the notebook

best_hyperparameters = {}
if RUN_SEARCH:
    from exist2026.fusion.hpo import optimize

    def objective_for(subtask_key, gate):
        def train_fn(hyperparameters, trial):
            return train_task(
                subtask_key,
                config=config,
                physio_in=experiment.physio_in,
                train_loader=experiment.train_loader(
                    subtask_key, config.training.merged(hyperparameters)["batch_size"], DEVICE
                ),
                val_loader=experiment.val_loader(device=DEVICE),
                gold=experiment.gold(subtask_key),
                checkpoint_name=f"_optuna_{subtask_key}.pt",
                hyperparameters=hyperparameters,
                gate=gate,
                trial=trial,
                device=DEVICE,
                verbose=False,
            ).best_metric

        return train_fn

    k1, k2, k3 = config.subtask_keys
    best_hyperparameters[k1] = optimize(k1, objective_for(k1, None), config=config).best_params

## 6 · Train the three subtasks and look at the curves

The gate for x.2 / x.3 is the *saved best* x.1 checkpoint, not the model object
left in memory after training — those differ whenever the last epoch was not
the best one.

In [ ]:
from exist2026.fusion.training import load_checkpoint, plot_curves
from exist2026.taxonomy import Subtask, subtask_of

histories, gate = {}, None
for subtask_key in config.subtask_keys:
    hyperparameters = {**best_hyperparameters.get(subtask_key, {}), "epochs": config.final_epochs}
    result = train_task(
        subtask_key,
        config=config,
        physio_in=experiment.physio_in,
        train_loader=experiment.train_loader(
            subtask_key, config.training.merged(hyperparameters)["batch_size"], DEVICE
        ),
        val_loader=experiment.val_loader(device=DEVICE),
        gold=experiment.gold(subtask_key),
        checkpoint_name=f"best_{subtask_key}.pt",
        hyperparameters=hyperparameters,
        gate=None if subtask_of(subtask_key) is Subtask.IDENTIFICATION else gate,
        device=DEVICE,
    )
    histories[subtask_key] = result.history
    print(f"{subtask_key}: best ICM-Soft-Norm = {result.best_metric:.4f}")
    if subtask_of(subtask_key) is Subtask.IDENTIFICATION:
        gate, _ = load_checkpoint(
            subtask_key, config.dirs.checkpoints / f"best_{subtask_key}.pt",
            config, experiment.physio_in, DEVICE,
        )

plot_curves(histories, f"{config.modality.value} - cross-attention fusion",
            config.dirs.figures / "train_curves.png")

## 7 · Refit and submit

With the hyperparameters chosen, refit on train+validation. There is no
held-out split left, so there is no checkpoint selection either: the epoch
count is fixed to the one validation already picked.

The physiological preprocessor is refitted on train+validation too — safe here,
and only here, because validation has already done its job and the test split
still contributes nothing to the statistics.

In [ ]:
from exist2026.fusion.training import predict_soft, refit_on_all
from exist2026.submission import package

preprocessor = experiment.full_preprocessor()
test_loader, test_ids = experiment.test_loader(preprocessor, device=DEVICE)

if test_loader is None:
    print("no test split configured; skipping the submission")
else:
    models = {
        subtask_key: refit_on_all(
            subtask_key,
            config=config,
            physio_in=experiment.physio_in,
            loader=experiment.full_loader(
                subtask_key,
                config.training.merged(best_hyperparameters.get(subtask_key, {}))["batch_size"],
                preprocessor,
                DEVICE,
            ),
            hyperparameters=best_hyperparameters.get(subtask_key, {}),
            epochs=config.final_epochs,
            device=DEVICE,
        )
        for subtask_key in config.subtask_keys
    }

    gate_model = models[config.subtask_keys[0]]
    predictions = {}
    for subtask_key in config.subtask_keys:
        records = predict_soft(
            models[subtask_key], test_loader, subtask_key, DEVICE,
            gate=None if subtask_of(subtask_key) is Subtask.IDENTIFICATION else gate_model,
        )
        predictions[subtask_key] = {r["id"]: r["value"] for r in records}

    base = package(
        predictions,
        modality=config.modality,
        output_dir=config.dirs.submission,
        team_name=config.team_name,
        run_id=config.run_id,
        evaluation_context="soft",
    )
    print(f"ready to upload: {base}.zip")

## 8 · Ablations

The paper reports two negative results, and both are read off this pipeline
rather than a separate one:

- **Physiology gives no soft-metric gain**, and occasionally hurts fine-grained
  categorization. Reported as a null result rather than dropped.
- **A dense visual stream is redundant** wherever the verbalized `capsanocr`
  view is already present.

To reproduce an ablation, change one thing and rerun section 6. For a text-only
model, zero the physiological input; the cross-attention block still runs and
the residual connections let the text stream carry the prediction on its own.

In [ ]:
# Text-only ablation: replace the preprocessor with one that returns zeros.
# Everything else - split, loaders, training loop, metric - stays identical,
# which is what makes the comparison a fair one.
class ZeroPhysio:
    def __init__(self, reference):
        self.reference = reference

    def transform(self, values):
        return np.zeros_like(self.reference.transform(values))

# ablated = experiment.dataset(experiment.split.train, preprocessor=ZeroPhysio(experiment.preprocessor))